# Modelling


To Do

Failing on the on the train/test split need to investigate
Run models again
check output, is there an improvement or do we need to expand the feature set. 

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 
from market_ml.EDA import * 
from market_ml.feature_engineering import *
from market_ml.models import *

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import *
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit


# Data Loading 


In [2]:
stock_dl = DataLoader(tickers=TICKERS, period="10y", interval="1d", file_name='stock_data.parquet')
benchmark_dl = DataLoader(tickers=[BENCHMARK_TICKERS], period="10y", interval="1d", file_name='benchmark_data.parquet')

market_data = stock_dl.load_data()
benchmark_data = benchmark_dl.load_data()
sectors = market_data["Sector"].unique()

benchmark_data

Data loaded for ['AAPL', 'MSFT', 'NVDA', 'META', 'GOOGL', 'JPM', 'GS', 'BAC', 'MS', 'BLK', 'JNJ', 'UNH', 'ABBV', 'PFE', 'MRK', 'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'AMZN', 'COST', 'WMT', 'HD', 'MCD']
Data loaded for [['SPY', 'QQQ']]


Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,QQQ,NaN,110.121559,110.943924,109.644969,110.738333,27207500
1,2016-09-29,SPY,NaN,183.335754,185.206007,182.789198,184.804628,128070600
2,2016-09-30,QQQ,NaN,110.943939,111.271013,110.299131,110.411274,25543200
3,2016-09-30,SPY,NaN,184.719223,185.419493,183.916465,184.164118,117202900
4,2016-10-03,QQQ,NaN,110.785042,110.925211,110.355165,110.803729,19216400
...,...,...,...,...,...,...,...,...
5021,2026-09-25,SPY,NaN,771.349976,772.280029,766.289978,768.780029,36666700
5022,2026-09-28,QQQ,NaN,736.530029,741.419983,731.630005,740.460022,41716200
5023,2026-09-28,SPY,NaN,765.609985,769.539978,763.719971,768.349976,42296700
5024,2026-09-29,QQQ,NaN,737.229980,740.579895,736.000000,740.169983,9703756


In [3]:
engineer = FeatureEngineer(market_data, benchmark_data, filename="stock_data_with_features.parquet")
data  = engineer.load_features()

print(data.head())

Data loaded for Price       Date Ticker      Sector       Close        High         Low  \
0     2016-09-29   AAPL  Technology   25.653965   26.024437   25.567065   
1     2016-09-30   AAPL  Technology   25.852909   25.926088   25.567052   
2     2016-10-03   AAPL  Technology   25.731701   25.852906   25.676817   
3     2016-10-04   AAPL  Technology   25.841478   26.141056   25.756864   
4     2016-10-05   AAPL  Technology   25.852909   25.992407   25.770582   
...          ...    ...         ...         ...         ...         ...   
62820 2026-09-23    XOM      Energy  161.229996  162.119995  159.419998   
62821 2026-09-24    XOM      Energy  162.139999  164.910004  161.199997   
62822 2026-09-25    XOM      Energy  160.589996  161.789993  159.779999   
62823 2026-09-28    XOM      Energy  162.520004  163.539993  161.050003   
62824 2026-09-29    XOM      Energy  161.625000  161.639999  159.270004   

Price        Open     Volume  
0       25.878078  143548000  
1       25.717984  14

In [4]:
print(data.shape)

print(data[["Date", "Ticker"]].head())

print(data.sort_values(["Date", "Ticker"]).reset_index(drop=True))



(61200, 23)
Price       Date Ticker
60    2016-12-23   AAPL
61    2016-12-27   AAPL
62    2016-12-28   AAPL
63    2016-12-29   AAPL
64    2016-12-30   AAPL
Price       Date Ticker      Sector       Close        High         Low  \
0     2016-12-23   AAPL  Technology   26.783262   26.783262   26.569493   
1     2016-12-23   ABBV  Healthcare   41.735088   41.775258   41.152646   
2     2016-12-23   AMZN    Consumer   38.029499   38.325001   37.899502   
3     2016-12-23    BAC  Financials   18.145319   18.185463   18.008828   
4     2016-12-23    BLK  Financials  303.477539  305.916442  302.383981   
...          ...    ...         ...         ...         ...         ...   
61195 2026-09-22    PFE  Healthcare   27.930000   28.000000   27.490000   
61196 2026-09-22    SLB      Energy   52.119999   52.910000   50.880001   
61197 2026-09-22    UNH  Healthcare  372.950012  381.390015  372.410004   
61198 2026-09-22    WMT    Consumer  110.120003  110.190002  107.610001   
61199 2026-09-22   

In [5]:
print("Start:", data["Date"].min())
print("End:  ", data["Date"].max())

Start: 2016-12-23 00:00:00
End:   2026-09-22 00:00:00


# Splitting Data into Train and Test 

In [6]:
data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_20D,Return_60D,Volatility_20D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D
60,2016-12-23,AAPL,Technology,26.783262,26.783262,26.569493,26.569493,56998000.0,0.000666,0.001464,...,0.042312,0.044020,0.007573,26.066441,25.910281,1.027500,1.033692,123165640.0,0.462775,-0.003175
61,2016-12-27,AAPL,Technology,26.953356,27.077480,26.776363,26.783258,73187600.0,0.005158,0.002481,...,0.051000,0.042566,0.007566,26.131836,25.911342,1.031438,1.040215,121386220.0,0.602932,-0.010575
62,2016-12-28,AAPL,Technology,26.838427,27.128049,26.709704,27.013119,83623600.0,-0.007780,-0.008264,...,0.047551,0.043010,0.007681,26.192749,25.910471,1.024651,1.035814,119861640.0,0.697668,-0.001285
63,2016-12-29,AAPL,Technology,26.831528,26.918874,26.755674,26.767166,60158000.0,-0.001418,-0.000223,...,0.056189,0.038312,0.007284,26.264121,25.909827,1.021604,1.035573,115637080.0,0.520231,0.010109
64,2016-12-30,AAPL,Technology,26.622362,26.939568,26.532717,26.813146,122345200.0,-0.010275,-0.003655,...,0.057814,0.029763,0.007157,26.336872,25.906601,1.010840,1.027629,114336960.0,1.070041,0.027370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62815,2026-09-16,XOM,Energy,163.320007,167.500000,162.559998,167.229996,13437800.0,0.000255,-0.004410,...,-0.013530,0.187098,0.016501,162.861002,156.272881,1.002818,1.045095,13751285.0,0.977203,-0.012797
62816,2026-09-17,XOM,Energy,163.270004,163.389999,161.449997,161.830002,10976500.0,0.017312,0.011339,...,-0.009104,0.176034,0.016472,162.786002,156.733840,1.002973,1.041702,13617665.0,0.806049,-0.006921
62817,2026-09-18,XOM,Energy,163.539993,163.539993,161.309998,162.500000,37640600.0,0.006319,0.001287,...,-0.015709,0.202330,0.016353,162.655502,157.273127,1.005438,1.039847,14809545.0,2.541645,-0.018038
62818,2026-09-21,XOM,Energy,158.300003,161.949997,157.940002,160.960007,15752500.0,0.028821,0.015505,...,-0.041245,0.158306,0.017772,162.315002,157.679397,0.975264,1.003936,14881455.0,1.058532,0.026658


In [7]:
data = data.drop(columns=[col for col in data.columns if "Target" in col]) #target was created for categorical, we don't want these in the regression
train_data, test_data, split_date = split_chronological_data(data)

data["Ticker"].unique()
print(train_data)


Empty DataFrame
Columns: [Date, Ticker, Sector, Close, High, Low, Open, Volume, QQQ_Return, SPY_Return, Daily_Return, Return_5D, Return_10D, Return_20D, Return_60D, Volatility_20D, SMA_20D, SMA_50D, Price_to_SMA_20D, Price_to_SMA_50D, Volume_Average_20D, Relative_Volume_20D, Future_Return_5D]
Index: []

[0 rows x 23 columns]


In [8]:
data.columns

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D', 'Return_10D',
       'Return_20D', 'Return_60D', 'Volatility_20D', 'SMA_20D', 'SMA_50D',
       'Price_to_SMA_20D', 'Price_to_SMA_50D', 'Volume_Average_20D',
       'Relative_Volume_20D', 'Future_Return_5D'],
      dtype='str', name='Price')

In [9]:
data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_20D,Return_60D,Volatility_20D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D
60,2016-12-23,AAPL,Technology,26.783262,26.783262,26.569493,26.569493,56998000.0,0.000666,0.001464,...,0.042312,0.044020,0.007573,26.066441,25.910281,1.027500,1.033692,123165640.0,0.462775,-0.003175
61,2016-12-27,AAPL,Technology,26.953356,27.077480,26.776363,26.783258,73187600.0,0.005158,0.002481,...,0.051000,0.042566,0.007566,26.131836,25.911342,1.031438,1.040215,121386220.0,0.602932,-0.010575
62,2016-12-28,AAPL,Technology,26.838427,27.128049,26.709704,27.013119,83623600.0,-0.007780,-0.008264,...,0.047551,0.043010,0.007681,26.192749,25.910471,1.024651,1.035814,119861640.0,0.697668,-0.001285
63,2016-12-29,AAPL,Technology,26.831528,26.918874,26.755674,26.767166,60158000.0,-0.001418,-0.000223,...,0.056189,0.038312,0.007284,26.264121,25.909827,1.021604,1.035573,115637080.0,0.520231,0.010109
64,2016-12-30,AAPL,Technology,26.622362,26.939568,26.532717,26.813146,122345200.0,-0.010275,-0.003655,...,0.057814,0.029763,0.007157,26.336872,25.906601,1.010840,1.027629,114336960.0,1.070041,0.027370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62815,2026-09-16,XOM,Energy,163.320007,167.500000,162.559998,167.229996,13437800.0,0.000255,-0.004410,...,-0.013530,0.187098,0.016501,162.861002,156.272881,1.002818,1.045095,13751285.0,0.977203,-0.012797
62816,2026-09-17,XOM,Energy,163.270004,163.389999,161.449997,161.830002,10976500.0,0.017312,0.011339,...,-0.009104,0.176034,0.016472,162.786002,156.733840,1.002973,1.041702,13617665.0,0.806049,-0.006921
62817,2026-09-18,XOM,Energy,163.539993,163.539993,161.309998,162.500000,37640600.0,0.006319,0.001287,...,-0.015709,0.202330,0.016353,162.655502,157.273127,1.005438,1.039847,14809545.0,2.541645,-0.018038
62818,2026-09-21,XOM,Energy,158.300003,161.949997,157.940002,160.960007,15752500.0,0.028821,0.015505,...,-0.041245,0.158306,0.017772,162.315002,157.679397,0.975264,1.003936,14881455.0,1.058532,0.026658


In [10]:
features = feature_columns = [
    "QQQ_Return",
    "SPY_Return",
    "Daily_Return",
    "Return_5D",
    "Return_10D",
    "Return_20D",
    "Return_60D",
    "Volatility_20D",
    "Price_to_SMA_20D",
    "Price_to_SMA_50D",
    "Relative_Volume_20D",
]

target = "Future_Return_5D" #target is the 5 day return. 
features

['QQQ_Return',
 'SPY_Return',
 'Daily_Return',
 'Return_5D',
 'Return_10D',
 'Return_20D',
 'Return_60D',
 'Volatility_20D',
 'Price_to_SMA_20D',
 'Price_to_SMA_50D',
 'Relative_Volume_20D']

In [21]:
train_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_20D,Return_60D,Volatility_20D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D


In [ ]:
x_train, y_train = train_data[features], train_data[target]
x_test, y_test = test_data[features], test_data[target]


60      -0.003175
61      -0.010575
62      -0.001285
63       0.010109
64       0.027370
           ...   
62815   -0.012797
62816   -0.006921
62817   -0.018038
62818    0.026658
62819    0.018367
Name: Future_Return_5D, Length: 61200, dtype: float64

# Baseline Model

want random model to compare to. 

In [14]:
print(y_train)

Series([], Name: Future_Return_5D, dtype: float64)


In [15]:
model_comparison_df = pd.DataFrame(columns=["Model", "MAE", "RMSE", "R2"]) 

In [16]:
baseline_prediction = y_train.mean()
baseline_y_pred = [baseline_prediction] * len(y_test)
print(f"Baseline prediction: {np.round(baseline_prediction,4)}")

Baseline prediction: nan


In [17]:
model_comparison_df = pd.DataFrame(columns=["Model", "MAE", "RMSE", "R2"]) 

baseline_mae = mean_absolute_error(y_test, baseline_y_pred)
baseline_rmse = root_mean_squared_error(y_test, baseline_y_pred)
baseline_r2 = r2_score(y_test, baseline_y_pred)

print(f"Baseline Mean absolute error is {np.round(baseline_mae,4)}")
print(f"Baseline Root mean squared error is {np.round(baseline_rmse,4)}")
print(f"Baseline R-squared is {np.round(baseline_r2,4)}")

baseline_metrics = ["Baseline Metrics", np.round(baseline_mae,4), np.round(baseline_rmse,4), np.round(baseline_r2,4)]

if baseline_metrics[0] not in model_comparison_df.columns:
    model_comparison_df.loc[len(model_comparison_df)] = baseline_metrics
model_comparison_df


ValueError: Input contains NaN.

# Linear Regression

Not using standard scaler initially  

In [ ]:
linearModel = LinearRegression()

linearModel.fit(x_train, y_train)

y_pred = linearModel.predict(x_test)

In [ ]:
MAE = mean_absolute_error(y_test, y_pred) #average size of the prediction error
RMSE = root_mean_squared_error(y_test, y_pred) #how far the predictions are from the actual values, penalises large errors.
R2 = r2_score(y_test, y_pred) #how much variation in the target variable the model predicts (1.0 is perfect).

print(f"Linear Regression Mean absolute error is {np.round(MAE,4)}")
print(f"Linear Regression Root mean squared error is {np.round(RMSE,4)}")
print(f"Linear Regression R-squared is {np.round(R2,4)}")

linear_regression_metrics = ["Linear Regression", np.round(MAE,4), np.round(RMSE,4), np.round(R2, 4)]

if linear_regression_metrics[0] not in model_comparison_df.columns:
    model_comparison_df.loc[len(model_comparison_df)] = linear_regression_metrics

model_comparison_df

Linear Regression Mean absolute error is 0.0306
Linear Regression Root mean squared error is 0.0416
Linear Regression R-squared is 0.0002


,Model,MAE,RMSE,R2
0,Baseline Metrics,0.0306,0.0416,-0.0001
1,Linear Regression,0.0306,0.0416,0.0002


Looks like our initial linear regression is pretty close to baseline. 

# Random Forest

In [ ]:
ntrees = 100
max_depth = 5 #default is None which I don't want to use at it'll overfit
random_state = 42 #for reproducability 
njobs = -1 #allowing it to use all available cores on my computer 
rf_model = RandomForestRegressor(n_estimators=ntrees, max_depth=max_depth, random_state=random_state, n_jobs=njobs)

In [ ]:
rf_model.fit(x_train, y_train)
y_pred = rf_model.predict(x_test)

In [ ]:
MAE = mean_absolute_error(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)

print(f"Mean absolute error is {np.round(MAE,4)}")
print(f"Root mean squared error is {np.round(RMSE,4)}")
print(f"R-squared is {np.round(R2,4)}")

random_forest_metrics = ["Random Forest", np.round(MAE,4), np.round(RMSE,4), np.round(R2, 4)]

if random_forest_metrics[0] not in model_comparison_df.columns:
    model_comparison_df.loc[len(model_comparison_df)] = random_forest_metrics
    model_comparison_df

model_comparison_df

Mean absolute error is 0.0308
Root mean squared error is 0.042
R-squared is -0.0184


,Model,MAE,RMSE,R2
0,Baseline Metrics,0.0306,0.0416,-0.0001
1,Linear Regression,0.0306,0.0416,0.0002
2,Random Forest,0.0308,0.0420,-0.0184


# Random Forest Hyperparameter Tuning

In [ ]:
rf_Model = RandomForestRegressor(random_state=42)

cv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "n_estimators":[50, 100, 200, 300, 400],
    "max_depth": [3, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 5],
}

grid_search = GridSearchCV(rf_Model,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)



In [ ]:
grid_search.fit(x_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'max_depth': 3, 'min_samples_leaf': 2, 'n_estimators': 50}
-0.03137097589499922


In [ ]:
best_parameters = grid_search.best_params_

rf_Model = RandomForestRegressor(n_estimators=best_parameters["n_estimators"], max_depth=best_parameters["max_depth"], min_samples_leaf=best_parameters["min_samples_leaf"], random_state=42)

rf_Model.fit(x_train, y_train)
y_pred = rf_Model.predict(x_test)




In [ ]:
MAE = mean_absolute_error(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)

print(f"Tuned Random Forest Mean absolute error is {np.round(MAE,4)}")
print(f"Tuned Random Forest Root mean squared error is {np.round(RMSE,4)}")
print(f"Tuned Random R-squared is {np.round(R2,4)}")

tuned_random_forest_metrics = ["Tuned Random Forest", np.round(MAE,4), np.round(RMSE, 4), np.round(R2, 4)]

if tuned_random_forest_metrics[0] not in model_comparison_df["Model"]:
    model_comparison_df.loc[len(model_comparison_df)] = tuned_random_forest_metrics
    model_comparison_df

model_comparison_df

Tuned Random Forest Mean absolute error is 0.0307
Tuned Random Forest Root mean squared error is 0.0417
Tuned Random R-squared is -0.0056


,Model,MAE,RMSE,R2
0,Baseline Metrics,0.0306,0.0416,-0.0001
1,Linear Regression,0.0306,0.0416,0.0002
2,Random Forest,0.0308,0.0420,-0.0184
3,Tuned Random Forest,0.0307,0.0417,-0.0056


Nothing meaningful predicted in linear regression or random forest compared to baseline. 

# Feature Importance

In [ ]:
rf_Model.feature_importances_

feature_importance = pd.DataFrame({"Features": features, "Importance":rf_Model.feature_importances_})

feature_importance = feature_importance.sort_values("Importance", ascending=False)

feature_importance

,Features,Importance
1,SPY_Return,0.293294
7,Volatility_20D,0.218833
0,QQQ_Return,0.198718
3,Return_5D,0.170787
4,Return_10D,0.036842
8,Price_to_SMA_20D,0.030339
6,Return_60D,0.027767
10,Relative_Volume_20D,0.011251
5,Return_20D,0.008184
9,Price_to_SMA_50D,0.002270


In [ ]:

fig = px.bar(feature_importance, x="Importance", y="Features", orientation="h", title="Random Forest Feature Importance")

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [ ]:
from sklearn.inspection import permutation_importance

permutation = permutation_importance(
    rf_Model,
    x_test,
    y_test,
    scoring="r2",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

In [ ]:
permutation_importance_df = pd.DataFrame({
    "Features": features,
    "Importance": permutation.importances_mean
}).sort_values(
    "Importance",
    ascending=False
)

permutation_importance_df

,Features,Importance
1,SPY_Return,0.006062
0,QQQ_Return,0.000996
3,Return_5D,0.000146
2,Daily_Return,-0.000010
6,Return_60D,-0.000020
5,Return_20D,-0.000080
8,Price_to_SMA_20D,-0.000139
10,Relative_Volume_20D,-0.000197
9,Price_to_SMA_50D,-0.000352
4,Return_10D,-0.000538


In [ ]:
fig = px.bar(permutation_importance_df, x="Importance", y="Features", orientation="h", title="Permutation Feature Importance")

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

# Let's compare the returns

In [ ]:
rf_predictions = rf_Model.predict(x_test)

predictions_df = test_data[["Date","Ticker", "Sector", "Future_Return_5D"]].copy()

predictions_df["Predicted_Return"] = rf_predictions

predictions_df

Price,Date,Ticker,Sector,Future_Return_5D,Predicted_Return
50450,2024-10-08,AAPL,Technology,0.035789,0.004029
50451,2024-10-08,ABBV,Healthcare,0.001980,0.004029
50452,2024-10-08,AMZN,Consumer,0.027200,0.004029
50453,2024-10-08,BAC,Financials,0.055347,0.004029
50454,2024-10-08,BLK,Financials,0.061039,0.004029
...,...,...,...,...,...
62695,2026-09-22,PFE,Healthcare,0.014143,0.004029
62696,2026-09-22,SLB,Energy,-0.045568,0.004029
62697,2026-09-22,UNH,Healthcare,0.001073,0.004029
62698,2026-09-22,WMT,Consumer,-0.039094,0.004029


In [ ]:
predictions_df[["Future_Return_5D", "Predicted_Return"]].describe()

Price,Future_Return_5D,Predicted_Return
count,12250.000000,12250.000000
mean,0.003635,0.003843
std,0.041577,0.004283
min,-0.290490,-0.077548
25%,-0.019951,0.004029
50%,0.003984,0.004029
75%,0.026730,0.004029
max,0.253277,0.026399


In [ ]:
predictions_df[["Future_Return_5D", "Predicted_Return"]].head()

Price,Future_Return_5D,Predicted_Return
50450,0.035789,0.004029
50451,0.001980,0.004029
50452,0.027200,0.004029
50453,0.055347,0.004029
50454,0.061039,0.004029


In [ ]:
fig = px.scatter(predictions_df, x="Date", y=["Future_Return_5D", "Predicted_Return"], title="Predicted and Actual Returns")
fig.update_layout(yaxis_title="Return")
fig.show()

predicted returns less variable and smaller than actual 5 day 

# By Stock Breakdown, does it perform badly for all of the stocks?

In [ ]:
predictions_df

Price,Date,Ticker,Sector,Future_Return_5D,Predicted_Return
50450,2024-10-08,AAPL,Technology,0.035789,0.004029
50451,2024-10-08,ABBV,Healthcare,0.001980,0.004029
50452,2024-10-08,AMZN,Consumer,0.027200,0.004029
50453,2024-10-08,BAC,Financials,0.055347,0.004029
50454,2024-10-08,BLK,Financials,0.061039,0.004029
...,...,...,...,...,...
62695,2026-09-22,PFE,Healthcare,0.014143,0.004029
62696,2026-09-22,SLB,Energy,-0.045568,0.004029
62697,2026-09-22,UNH,Healthcare,0.001073,0.004029
62698,2026-09-22,WMT,Consumer,-0.039094,0.004029


In [ ]:
stock_performance = predictions_df.groupby("Ticker").apply(lambda x: pd.Series({"MAE": mean_absolute_error(x["Future_Return_5D"], x["Predicted_Return"]),
                                                            "R2": r2_score(x["Future_Return_5D"], x["Predicted_Return"]),
                                                            "Mean Actual": x["Future_Return_5D"].mean(),
                                                            "Mean Predicted": x['Predicted_Return'].mean(),
                                                            "Baseline MAE":mean_absolute_error(x["Future_Return_5D"], [x["Future_Return_5D"].mean()]*len(x)),
                                                            "Baseline R2":r2_score(x["Future_Return_5D"], [x["Future_Return_5D"].mean()]*len(x)),
                                                            }
                                                            ) 
                                                        )

stock_performance["MAE Improvement"] = stock_performance["Baseline MAE"] - stock_performance["MAE"]

stock_performance

,MAE,R2,Mean Actual,Mean Predicted,Baseline MAE,Baseline R2,MAE Improvement
Ticker,,,,,,,
AAPL,0.031303,-0.006953,0.004934,0.004073,0.031281,0.0,-0.000022
ABBV,0.028527,0.019168,0.004647,0.003658,0.028737,0.0,0.000211
AMZN,0.035186,-0.006136,0.004073,0.003690,0.035092,0.0,-0.000094
BAC,0.027612,-0.014567,0.004328,0.003687,0.027406,0.0,-0.000206
BLK,0.027552,-0.013625,0.002104,0.003699,0.027637,0.0,0.000085
COP,0.035051,0.010575,0.003071,0.003750,0.035168,0.0,0.000118
COST,0.023105,-0.036357,0.000770,0.003725,0.022643,0.0,-0.000462
CVX,0.026337,0.032702,0.004654,0.003691,0.026601,0.0,0.000264
EOG,0.031856,0.013278,0.002207,0.003693,0.031981,0.0,0.000125


positive MAE difference - improvement from baseline to model. Only a few stocks have an improvement and even then it is tiny. Overall the random forest provides little improvement over the baseline historical mean when predicting the 5-day equity return. Some variability over the universe but overall not great. Predictions very close to the mean suggesting that the current feature set provides limited imformation about the short term returns size. 

In [ ]:
# have a quick look at the sector breakdown
sector_performance = predictions_df.groupby("Sector").apply(lambda x: pd.Series({"MAE": mean_absolute_error(x["Future_Return_5D"], x["Predicted_Return"]),
                                                            "R2": r2_score(x["Future_Return_5D"], x["Predicted_Return"]),
                                                            "Mean Actual": x["Future_Return_5D"].mean(),
                                                            "Mean Predicted": x['Predicted_Return'].mean(),
                                                            "Baseline MAE":mean_absolute_error(x["Future_Return_5D"], [x["Future_Return_5D"].mean()]*len(x)),
                                                            "Baseline R2":r2_score(x["Future_Return_5D"], [x["Future_Return_5D"].mean()]*len(x)),
                                                            }
                                                            ) 
                                                        )

sector_performance["MAE Improvement"] = sector_performance["Baseline MAE"] - sector_performance["MAE"]

sector_performance.sort_values("MAE Improvement")

,MAE,R2,Mean Actual,Mean Predicted,Baseline MAE,Baseline R2,MAE Improvement
Sector,,,,,,,
Consumer,0.026389,-0.024372,0.000886,0.003712,0.026077,0.0,-0.000312
Financials,0.028807,-0.014666,0.005327,0.003695,0.028637,0.0,-0.000169
Healthcare,0.029941,-0.011321,0.003006,0.003999,0.029887,0.0,-0.000054
Technology,0.036401,-0.006464,0.005494,0.004021,0.036404,0.0,0.000003
Energy,0.031812,0.014198,0.003462,0.003789,0.031947,0.0,0.000135


Minimal improvements from baseline in energy and technology, rest are worse that the baseline.